In [0]:
from pyspark.sql.functions import col, to_timestamp

# 1. Read the raw Bronze orders table using Unity Catalog's 3-part namespace
bronze_orders = spark.table("workspace.bronze.orders_dataset")

# 2. Clean the data: Drop null primary keys and cast string dates to proper Timestamps
silver_orders = bronze_orders \
    .dropna(subset=["order_id"]) \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))

# 3. Write to the Silver schema as a managed Delta table
silver_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.orders")

# 4. View the new schema to prove the data types changed
silver_orders.printSchema()
display(silver_orders.limit(5))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-10-23T18:03:28.213Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_orders_dataset.csv
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-10-23T18:03:28.213Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_orders_dataset.csv
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-10-23T18:03:28.213Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_orders_dataset.csv
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-10-23T18:03:28.213Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_orders_dataset.csv
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-10-23T18:03:28.213Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_orders_dataset.csv


In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# 1. Read the raw Bronze order items table
bronze_order_items = spark.table("workspace.bronze.order_items_dataset")

# 2. Clean the data: Deduplicate and cast financial columns to Double
silver_order_items = bronze_order_items \
    .dropDuplicates() \
    .withColumn("price", col("price").cast(DoubleType())) \
    .withColumn("freight_value", col("freight_value").cast(DoubleType()))

# 3. Write to the Silver schema as a managed Delta table
silver_order_items.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.order_items")

# 4. View the new schema and data
silver_order_items.printSchema()
display(silver_order_items.limit(5))

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,source_file
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-10-23T18:03:14.173Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_items_dataset.csv
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-10-23T18:03:14.173Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_items_dataset.csv
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-10-23T18:03:14.173Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_items_dataset.csv
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-10-23T18:03:14.173Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_items_dataset.csv
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,2026-10-23T18:03:14.173Z,dbfs:/Volumes/workspace/bronze/raw_landing/olist_order_items_dataset.csv
